# Snowshoe hares at Bonanza Creek Experimental Forest
Week 3 Discussion Section

## Archive exploration

### Data Description
This data was produced by researchers as part of the Long Term Ecological Research Program. It consists of capture-recapture studies of snowshoe hare populations in Alaska from 1999 to 2002 for the purposes of monitoring population fluctuations. Each row in the column represents a hare capture observation. Hare weights are measured in grams. There is no key for how missing data is encoded.

### Citation:
Kielland, K., F.S. Chapin, R.W. Ruess, and Bonanza Creek LTER. 2017. Snowshoe hare physical data in Bonanza Creek Experimental Forest: 1999-Present ver 22. Environmental Data Initiative. https://doi.org/10.6073/pasta/03dce4856d79b91557d8e6ce2cbcdc14 (Accessed 2026-10-15).

### Date Accessed:
10/15/26

### Link to dataset:
https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-bnz.55.22

![image of snowshoe hare](https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1920px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20170313021652)

### What are some possible issues with the data?
How will missing data be indicated?
Are the amount of observations comparable throughout the years?

## Data loading and preliminary exploration
- What are the dimensions of the data frame and what are the data types of the columns? Do the data types match what you would expect from each column?
- Are there any columns that have a significant number of NA values?
- What are the minimum and maximum values for the weight and hind feet measurements?
- What are the unique values for some of the categorical columns?
- Did any of the issues you anticipated in 1c show up?

In [28]:
# Import library and read data
import numpy as np
import pandas as pd
hares = pd.read_csv("data/55_Hare_Data_2012.txt")

In [2]:
# Find dimmensions of dataframe
hares.shape

# Find data types in columns
hares.dtypes

date           object
time           object
grid           object
trap           object
l_ear          object
r_ear          object
sex            object
age            object
weight        float64
hindft        float64
notes          object
b_key         float64
session_id      int64
study          object
dtype: object

Note: Date and time are objects not datetimes

In [3]:
# Count NAs in each column
hares.isna().sum()

date             0
time          3116
grid             0
trap            12
l_ear           48
r_ear          169
sex            352
age           2111
weight         535
hindft        1747
notes         3137
b_key           47
session_id       0
study          163
dtype: int64

In [4]:
# Check min and max weight
print(hares['weight'].max())

print(hares['weight'].min())

2365.0
0.0


In [5]:
# Check min and max hindfoot measurement
print(hares['hindft'].max())

print(hares['hindft'].min())

160.0
60.0


In [6]:
# Check unique values of grid column
hares['grid'].unique()

array(['bonrip', 'bonbs', 'Bonbs', 'bonmat', 'bonmat '], dtype=object)

In [7]:
# Check unique values of sex column
hares['sex'].unique()

array([nan, 'M', 'F', '?', 'F?', 'M?', 'pf', 'm', 'f', 'f?', 'm?', 'f ',
       'm '], dtype=object)

In [8]:
# Check unique values of age column
hares['age'].unique()

array([nan, 'J', 'A', 'a 1 yr.', 'a 3/4 yr.', 'a 1 yr', '1 yr', '1 yr.',
       '2 yrs.', '2 yrs', 'a 2 yrs.', '2.25 yrs', '3.5 yrs.', '3 yrs.',
       '2.5 yrs', '3.25 yrs.', 'A 1.5', '?', 'U', 'j', 'a', 'u', 'J 3/4',
       'A 3/4', 'A 1/2', '3/4/2013', '1/4/2013', '1/2/2013', '1', '1.25',
       '1.5'], dtype=object)

## Detect Messy Values

`sex` column allowed values and definitions:
| `sex` code  | Definition  |
| ----------- | ----------- |
| m           | male        |
| f           | female      |
| m?          | male not confirmed |

In [11]:
# Count instances of non-NA values
hares['sex'].value_counts()

sex
F     1161
M      730
f      556
m      515
?       40
F?      10
f        4
m        4
f?       3
M?       2
m?       2
pf       1
Name: count, dtype: int64

In [12]:
hares['sex'].value_counts(dropna=False)

sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64

There are many more unique values than the 'allowed' 3 values. Probably a result of different researchers contributing to the data. The 'repeat' values are likely due to spaces before or after

## Brainstorm how to clean `sex` column
1. remove spaces (str_strip)
2. make all lowercase
3. determine 'pf' value
4. 
5. 


## Clean values

In [32]:
# Remove spaces and uppercase
hares['sex_simple'] = hares['sex'].str.strip(' ').str.lower()
hares['sex_simple'].value_counts(dropna = False)

sex_simple
f      1721
m      1249
NaN     352
?        40
f?       13
m?        4
pf        1
Name: count, dtype: int64

In [33]:
# 
hares['sex_simple'] = hares['sex_simple'].replace('pf', 'f?')
hares['sex_simple'].value_counts()

sex_simple
f     1721
m     1249
?       40
f?      14
m?       4
Name: count, dtype: int64

In [36]:
hares['sex_simple'] = hares['sex_simple'].replace('?', np.NaN)
hares['sex_simple'].value_counts(dropna = False)

sex_simple
f      1721
m      1249
NaN     392
f?       14
m?        4
Name: count, dtype: int64

In [37]:
sex_map = {'f': 'female', 'm': 'male'}
hares['sex_simple'] = hares['sex_simple'].map(sex_map) # Default if doesnt match map, converts to NaN
hares['sex_simple'].value_counts(dropna = False)

sex_simple
female    1721
male      1249
NaN        410
Name: count, dtype: int64

## Calculate mean weight by sex

In [39]:
# Subset dataframe where weights are greater than 0
hares_valid = hares[hares['weight'] > 0]
hares_valid['weight'].min()

160.0

In [43]:
# Caluclate mean weights by sex
hares_valid['weight'].groupby(hares_valid['sex_simple']).mean()

sex_simple
female    1365.164792
male      1349.935542
Name: weight, dtype: float64